In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
%cd /content/
!rm -rf conflictbank
!git clone https://github.com/devltt404/conflictbank.git
%cd conflictbank

/content
Cloning into 'conflictbank'...
remote: Enumerating objects: 77, done.
remote: Counting objects: 100% (77/77), done.
remote: Compressing objects: 100% (56/56), done.
remote: Total 77 (delta 40), reused 40 (delta 18), pack-reused 0 (from 0)
Receiving objects: 100% (77/77), 46.05 KiB | 15.35 MiB/s, done.
Resolving deltas: 100% (40/40), done.
/content/conflictbank


In [3]:
!pip install -q -r requirements.txt
!pip install -q vllm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 82.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 159.1/159.1 kB 18.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 6.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.5/49.5 kB 5.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.4/84.4 kB 10.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 kB 4.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.4/60.4 kB 7.0 MB/s eta 0:00:00
ERROR: Ignored the following versions that require a different python version: 1.10.0 Requires-Python <3.12,>=3.8; 1.10.0rc1 Requires-Python <3.12,>=3.8; 1.10.0rc2 Requires-Python <3.12,>=3.8; 1.10.1 Requires-Python <3.12,>=3.8; 1.21.2 Requires-Python >=3.7,<3.11; 1.21.3 Requires-Python >=3.7,<3.11; 1.21.4 Requires

In [4]:
subset_size = 10000
# MODEL_FULL_PATH = "Qwen/Qwen2.5-7B"
MODEL_FULL_PATH = "HuggingFaceTB/SmolLM2-360M"
MODEL_SHORT_NAME = MODEL_FULL_PATH.split('/')[-1]
print(f"Model Full Path: {MODEL_FULL_PATH}")
print(f"Model Short Name: {MODEL_SHORT_NAME}")
print(f"Subset Size: {subset_size}")

Model Full Path: HuggingFaceTB/SmolLM2-360M
Model Short Name: SmolLM2-360M
Subset Size: 10000


In [7]:
from datasets import load_dataset
import json

dataset = load_dataset("Warrieryes/CB_qa", split="train")

subset = dataset.select(range(subset_size))

with open("cb_qa.jsonl", "w", encoding="utf-8") as f:
    for ex in subset:
        f.write(json.dumps(ex, ensure_ascii=False) + "\n")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


QA_dataset.json:   0%|          | 0.00/5.79G [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

In [8]:
!rm -rf prompts/

In [9]:
!mkdir prompts
!python prompt.py --in_file cb_qa.jsonl --out_dir prompts

In [10]:
!rm -rf runs/

In [11]:
!python inference.py \
  --model_path {MODEL_FULL_PATH} \
  --input_dir prompts \
  --out_dir runs \
  --logprobs 100

2026-04-08 22:05:09.065892: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-04-08 22:05:09.084824: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1775685909.109939    9092 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1775685909.117247    9092 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1775685909.134871    9092 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

In [ ]:
!python evaluate.py runs/{MODEL_SHORT_NAME} results

In [ ]:
import datetime
import glob
import os

# MODEL_SHORT_NAME is now defined in a previous cell

timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
zip_filename = f"{MODEL_SHORT_NAME}_{subset_size}_{timestamp}.zip"

# Find the current notebook file
notebook_name = "ConflictBank_Survey_E1.ipynb"
notebook_path = ""

# Search in common Colab Notebooks directory and current content directory
possible_paths = [
    f"/content/drive/MyDrive/Colab Notebooks/{notebook_name}",
    f"/content/{notebook_name}"
]

for path in possible_paths:
    if os.path.exists(path):
        notebook_path = path
        print(f"Found notebook: {notebook_path}")
        break

if not notebook_path:
    # Fallback to a broader search if not found in specific paths
    all_notebook_files = glob.glob('/content/**/*.ipynb', recursive=True)
    for nb_file in all_notebook_files:
        if os.path.basename(nb_file) == notebook_name:
            notebook_path = nb_file
            print(f"Found notebook: {notebook_path}")
            break

if not notebook_path:
    print(f"Could not find notebook file '{notebook_name}'.")


!rm -f "/content/{zip_filename}"

# Zip results folder and the notebook file if found
zip_command = f'zip -r "/content/{zip_filename}" /content/conflictbank/results/'
if notebook_path:
    zip_command += f' "{notebook_path}"'

!{zip_command}

In [ ]:
# Define the path where you want to save the file in your Google Drive
drive_output_path = '/content/drive/My Drive/Colab_Output'

# Create the directory in Drive if it doesn't exist
!mkdir -p "{drive_output_path}"

# Copy the zip file to Google Drive
!cp "/content/{zip_filename}" "{drive_output_path}"

The `file.zip` should now be available in your Google Drive at `My Drive/Colab_Output` (or the path you specified). You can check your Google Drive to confirm its presence.